## 1. Setup

In [2]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
load_dotenv()

from src.models_llm import (
    get_groq_client, get_mistral_client,
    ask_groq, ask_mistral,
    build_zero_shot_prompt, build_few_shot_prompt,
    normalize_prediction,
    build_support_set, run_experiment,
    evaluate, compare_experiments, confusion_df,
)

## 2. Carregar dados

In [3]:
df_train = pd.read_csv("../data/processed/dataset_combined.csv", sep=";")
df_test  = pd.read_csv("../data/validation/subm1_labels_revealed.csv", sep=";")

print("Treino:")
print(df_train['Label'].value_counts())
print("\nTeste:")
print(df_test['Label'].value_counts())

Treino:
Label
Anthropic    500
OpenAI       481
Google       480
Human        254
Name: count, dtype: int64

Teste:
Label
Human        34
Meta         18
Anthropic    17
Google       17
OpenAI       14
Name: count, dtype: int64


## 3. Clientes

In [4]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../src/.env")

print("GROQ:", bool(os.getenv("GROQ_API_KEY")))
print("MISTRAL:", bool(os.getenv("MISTRAL_API_KEY")))

groq_client    = get_groq_client()
mistral_client = get_mistral_client()

ask_groq_fn    = lambda prompt: ask_groq(prompt, client=groq_client)
ask_mistral_fn = lambda prompt: ask_mistral(prompt, client=mistral_client)

GROQ: True
MISTRAL: True


## 4. Support set (para few-shot)

In [5]:
support_1 = build_support_set(df_train, n_per_class=1)
support_2 = build_support_set(df_train, n_per_class=2)

pd.DataFrame(support_2)['Label'].value_counts()

Label
Google       2
Human        2
OpenAI       2
Anthropic    2
Name: count, dtype: int64

## 5. Zero-shot — Groq

In [ ]:
zs_groq_preds, zs_groq_raw = run_experiment(
    df_test, ask_groq_fn,
    build_zero_shot_prompt, normalize_prediction,
    sleep_seconds=0.5,
)
df_zs_groq = evaluate(zs_groq_preds, df_test, experiment_name="Groq zero-shot")

## 6. Few-shot — Groq (1 exemplo por classe)

In [ ]:
fs_groq_preds, fs_groq_raw = run_experiment(
    df_test, ask_groq_fn,
    build_few_shot_prompt, normalize_prediction,
    support_examples=support_1,
    sleep_seconds=2,
)
df_fs_groq = evaluate(fs_groq_preds, df_test, experiment_name="Groq few-shot 1x")

## 7. Few-shot — Groq (2 exemplos por classe)

In [ ]:
fs2_groq_preds, _ = run_experiment(
    df_test, ask_groq_fn,
    build_few_shot_prompt, normalize_prediction,
    support_examples=support_2,
    sleep_seconds=1,
)
df_fs2_groq = evaluate(fs2_groq_preds, df_test, experiment_name="Groq few-shot 2x")

## 8. Mistral Experiments

### 8.1 Zero-shot — Mistral

In [ ]:
zs_mistral_preds, zs_mistral_raw = run_experiment(
    df_test, ask_mistral_fn,
    build_zero_shot_prompt, normalize_prediction,
    sleep_seconds=5,
)
df_zs_mistral = evaluate(zs_mistral_preds, df_test, experiment_name="Mistral zero-shot")

### 8.2 Few-shot — Mistral (1 exemplo por classe)

In [ ]:
fs_mistral_preds, fs_mistral_raw = run_experiment(
    df_test, ask_mistral_fn,
    build_few_shot_prompt, normalize_prediction,
    support_examples=support_1,
    sleep_seconds=2,
)
df_fs_mistral = evaluate(fs_mistral_preds, df_test, experiment_name="Mistral few-shot 1x")

### 8.3 Few-shot — Mistral (2 exemplos por classe)

In [7]:
fs2_mistral_preds, _ = run_experiment(
    df_test, ask_mistral_fn,
    build_few_shot_prompt, normalize_prediction,
    support_examples=support_2,
    sleep_seconds=1,
)
df_fs2_mistral = evaluate(fs2_mistral_preds, df_test, experiment_name="Mistral few-shot 2x")

OpenAI
OpenAI
OpenAI
OpenAI
Human
OpenAI
Human
Human
Human
Anthropic
  [10/100] done
OpenAI
Human
Human
Human
Human
Human
Human
Human
Human
Human
  [20/100] done
Human
OpenAI
OpenAI
Human
OpenAI
Human
Human
Human
Human
Human
  [30/100] done
Anthropic
OpenAI
Human
Human
Human
Human
Human
Human
Human
Google
  [40/100] done
Human
OpenAI
Human
Human
OpenAI
Human
Human
OpenAI
OpenAI
OpenAI
  [50/100] done
Human
OpenAI
Human
Human
Human
Google
Google
Human
Human
Human
  [60/100] done
OpenAI
Human
Google
OpenAI
OpenAI
Human
OpenAI
OpenAI
Human
Google
  [70/100] done
Google
Human
Human
Human
Human
Human
Human
Human
Human
OpenAI
  [80/100] done
Human
Human
OpenAI
Human
OpenAI
Google
Google
Human
Human
Human
  [90/100] done
Google
OpenAI
OpenAI
OpenAI
Human
Human
Google
Human
OpenAI
Human
  [100/100] done

  Mistral few-shot 2x
  Accuracy : 0.3600  (100/100 parseable)
  Null rate: 0.00%
              precision    recall  f1-score   support

   Anthropic       0.00      0.00      0.00        17
 

## 9. Comparação final

In [ ]:
summary = compare_experiments({
    # "Groq zero-shot":    zs_groq_preds,
    # "Groq few-shot 1x":  fs_groq_preds,
    "Groq few-shot 2x":  fs2_groq_preds,
    # "Mistral zero-shot": zs_mistral_preds,
    # "Mistral few-shot 1x":  fs_mistral_preds,
    "Mistral few-shot 2x":  fs2_mistral_preds,
}, df_test)

display(summary)
summary.plot(x="experiment", y="accuracy", kind="bar", figsize=(10,4))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 10. Matriz de confusão do melhor modelo

In [ ]:
cm = confusion_df(fs2_groq_preds, df_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion matrix — Groq few-shot 2x')
plt.show()

## 11. Guardar previsões

In [ ]:
df_test['predicted_label'] = fs2_groq_preds  # melhor modelo
os.makedirs("../data/predictions", exist_ok=True)
df_test.to_csv("../data/predictions/llm_groq_fewshot.csv", sep=";", index=False)